# A股次日涨跌方向预测 demo

用腾讯行情接口取数据（免费、无令牌、直连） → 特征工程 → LightGBM 分类 → walk-forward 回测。
预测周期用 `horizon` 控制（1=次日，2=后天），股票池在 `stocklook/config.py` 里配置。
`threshold` （涨跌幅阈值）用于把接近零的小幅波动当作中性（横盘）过滤掉，默认 `config.THRESHOLD`（0 = 不过滤）。
另外支持 `--llm` 实时快照 + DeepSeek 大模型分析（见 CLI：`python -m stocklook --stocks 600519 --llm`）。

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd

from stocklook import config
from stocklook.data import fetch_history
from stocklook.features import add_features, build_label
from stocklook.pipeline import run


In [ ]:
from datetime import datetime

symbol = config.DEFAULT_STOCKS[0]
horizon = config.HORIZON
threshold = config.THRESHOLD
end_date = datetime.today().strftime("%Y%m%d")

df = fetch_history(symbol, config.START_DATE, end_date, config.ADJUST)
df = add_features(df)
df["label"] = build_label(df, horizon, threshold)
df.tail()

In [ ]:
ax = df.plot(x="date", y="close", figsize=(12, 4), legend=False, title=f"{symbol} 前复权收盘价")
ax.set_ylabel("close")
plt.show()

In [ ]:
summary, importances = run(config.DEFAULT_STOCKS, horizon, threshold)
with pd.option_context("display.float_format", "{:.4f}".format):
    display(summary)


In [ ]:
imp_df = pd.DataFrame(importances).T
imp_df.head(5)

## 怎么看这些指标

- `up_rate`：样本里真实上涨的天数占比（基准线，预测"涨"的胜率要超过它才有意义）
- `accuracy`：整体准确率
- `auc`：排序能力（>0.5 说明模型有信息量）
- `win_rate`：模型预测"涨"时的命中率（对"给自己投资参考"最直接）

个股次日涨跌接近随机，长期 `accuracy` 约 50% 是正常的。重点看 `auc` 是否稳定大于 0.5、`win_rate` 是否超过 `up_rate`。

上传的 `imp_df` 是每只股票的特征重要性（按重要度排序），用于判断该涨/跌信号主要受哪些量价/技术指标驱动——帮助你在信任一个信号前先理解它。